# Research Ergonomics Quickstart

This notebook demonstrates the canonical quickstart path for the approved `Research Ergonomics` slice.


In [ ]:
from __future__ import annotations

from quantcraft.research import Strategy, qc, run_backtest, ta
from quantcraft.research.adapters.synthetic_events import OHLCVBar
from quantcraft.trading.domain.costs import CostConfig

rows = (
    OHLCVBar(timestamp=60, open=5.0, high=5.0, low=5.0, close=5.0, volume=1.0),
    OHLCVBar(timestamp=120, open=4.0, high=4.0, low=4.0, close=4.0, volume=1.0),
    OHLCVBar(timestamp=180, open=1.0, high=1.0, low=1.0, close=1.0, volume=1.0),
    OHLCVBar(timestamp=240, open=10.0, high=10.0, low=10.0, close=10.0, volume=1.0),
    OHLCVBar(timestamp=300, open=11.0, high=11.0, low=11.0, close=11.0, volume=1.0),
)

costs = CostConfig(tick_size=1.0, slippage_ticks=1.0, fee_rate=0.001)


In [ ]:
class SmaCrossStrategy(Strategy):
    def init(self) -> None:
        self.fast = ta.sma(self.data.close, length=2)
        self.slow = ta.sma(self.data.close, length=3)

    def on_bar(self, bar) -> None:
        if qc.crossover(self.fast, self.slow):
            self.buy(symbol=bar.symbol, quantity=1)
        elif qc.crossunder(self.fast, self.slow):
            self.sell(symbol=bar.symbol, quantity=1)


sma_result = run_backtest(
    symbol="BTC/USDT",
    bar_type="time",
    bar_spec="1m",
    rows=rows,
    strategy=SmaCrossStrategy(),
    initial_cash=1_000.0,
    costs=costs,
)

sma_result.summary
sma_result.trade_log


In [ ]:
class Rsi3070Strategy(Strategy):
    def init(self) -> None:
        self.rsi = ta.rsi(self.data.close, length=2)

    def on_bar(self, bar) -> None:
        if not qc.is_na(self.rsi[0]) and self.rsi[0] < 30:
            self.buy(symbol=bar.symbol, quantity=1)
        elif not qc.is_na(self.rsi[0]) and self.rsi[0] > 70:
            self.sell(symbol=bar.symbol, quantity=1)


rsi_result = run_backtest(
    symbol="BTC/USDT",
    bar_type="time",
    bar_spec="1m",
    rows=rows,
    strategy=Rsi3070Strategy(),
    initial_cash=1_000.0,
    costs=costs,
)

rsi_result.equity_curve
rsi_result.trade_log
